# MSF News Classification Pipeline

This notebook collects humanitarian news articles, enriches them with countries, sentiment, and rule-based labels, then trains three models:

1. TF-IDF + Naive Bayes
2. spaCy vectors + Logistic Regression
3. DistilBERT transformer classifier

Each model prints basic evaluation metrics and a confusion matrix when enough labelled data is available.

## 0. Environment setup

This section prints the Python version and shows the packages needed to run the notebook. The install commands are commented out so the notebook does not reinstall packages every time it runs.

In [1]:
import sys

print("Python version:", sys.version)

# !pip install requests pandas numpy spacy pycountry spacytextblob scikit-learn transformers datasets accelerate torch
# !python -m spacy download en_core_web_sm

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


## 1. Imports

We import all libraries used for data collection, preprocessing, NLP, traditional machine learning, and transformer training.

In [1]:
import os
import requests
import pandas as pd
import numpy as np

from datetime import datetime, timedelta

import spacy
import pycountry
from spacytextblob.spacytextblob import SpacyTextBlob

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Load spaCy model

spaCy is used for named entity recognition, country detection, text vectors, and sentiment through `spacytextblob`.

In [2]:
nlp = spacy.load("en_core_web_sm")

if "spacytextblob" not in nlp.pipe_names:
    nlp.add_pipe("spacytextblob")

print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'spacytextblob']


## 3. Constants and country configuration

The country list focuses the dataset on countries where MSF-related humanitarian monitoring may be relevant. Aliases help normalize names like `DRC`, `Gaza`, and `West Bank`.

In [3]:
BASE_URL = "https://newsapi.org/v2/everything"

MSF_COUNTRIES = {
    "Afghanistan", "Bangladesh", "Burundi", "Central African Republic",
    "Chad", "Democratic Republic of the Congo", "Ethiopia",
    "Haiti", "Iraq", "Lebanon", "Mali", "Myanmar", "Niger", "Nigeria",
    "Palestine", "Somalia", "South Sudan", "Sudan", "Syria", "Ukraine",
    "Yemen"
}

COUNTRY_ALIASES = {
    "DR Congo": "Democratic Republic of the Congo",
    "DRC": "Democratic Republic of the Congo",
    "Congo": "Democratic Republic of the Congo",
    "Gaza": "Palestine",
    "West Bank": "Palestine",
}

## 4. Helper functions

These functions are reused throughout the notebook:

- `sentiment`: assigns Positive, Negative, or Neutral sentiment
- `classify_rule_based`: creates initial rule-based topic labels
- `detect_countries`: extracts and normalizes country mentions
- `safe_train_test_split`: avoids errors when there is too little data for stratified splitting
- `print_model_report`: prints metrics and confusion matrix

In [4]:
def sentiment(text):
    """Return Positive, Negative, or Neutral sentiment using spaCyTextBlob polarity."""
    doc = nlp(str(text))
    polarity = doc._.blob.polarity

    if polarity > 0.1:
        return "Positive"
    elif polarity < -0.1:
        return "Negative"
    return "Neutral"


def classify_rule_based(text):
    """Assign a simple humanitarian category based on keyword rules."""
    text = str(text).lower()

    if any(k in text for k in ["war", "conflict", "violence", "attack", "fighting"]):
        return "Conflict"
    if any(k in text for k in ["hospital", "disease", "health", "outbreak", "vaccine", "cholera", "malaria"]):
        return "Health"
    if any(k in text for k in ["flood", "earthquake", "disaster", "drought"]):
        return "Disaster"
    if any(k in text for k in ["aid", "refugee", "humanitarian", "displacement", "displaced"]):
        return "Humanitarian"

    return "Other"


def detect_countries(text):
    """Detect countries from text and keep only countries in MSF_COUNTRIES."""
    doc = nlp(str(text))
    detected = set()

    for ent in doc.ents:
        if ent.label_ == "GPE":
            name = ent.text.strip()

            if name in COUNTRY_ALIASES:
                detected.add(COUNTRY_ALIASES[name])
            elif name in MSF_COUNTRIES:
                detected.add(name)
            else:
                try:
                    country = pycountry.countries.lookup(name)
                    if country.name in MSF_COUNTRIES:
                        detected.add(country.name)
                except LookupError:
                    pass

    return sorted(detected)


def safe_train_test_split(X, y, test_size=0.2, random_state=42):
    """Use stratified split when possible; otherwise fall back safely."""
    min_class_count = y.value_counts().min()
    stratify = y if min_class_count >= 2 else None

    return train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )


def print_model_report(y_true, y_pred, title="Model report"):
    """Print classification metrics and confusion matrix."""
    print("=" * 80)
    print(title)
    print("=" * 80)
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

## 5. Fetch articles from NewsAPI

This section queries NewsAPI for recent humanitarian, medical, conflict, and crisis-related articles. Replace `YOUR_API_KEY` or import your key from your local `nocommit.py` file.

In [5]:
try:
    from nocommit import NEWSAPI_KEY
except ImportError:
    NEWSAPI_KEY = "YOUR_API_KEY"

query = """
(humanitarian OR medical OR hospital OR disease OR outbreak OR vaccine OR malnutrition
OR refugee OR displacement OR conflict OR war OR violence OR crisis)
"""

params = {
    "q": query.strip(),
    "language": "en",
    "sortBy": "publishedAt",
    "from": (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d"),
    "apiKey": NEWSAPI_KEY,
    "pageSize": 100,
}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()

articles = response.json().get("articles", [])
print(f"Retrieved articles: {len(articles)}")

df = pd.DataFrame(articles)

if df.empty:
    raise ValueError("No articles returned from NewsAPI.")

df.head()

Retrieved articles: 93


,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'Dailymail.com'}",Nicholas Comino,Albanese responds after man allegedly opens fi...,Prime Minister Anthony Albanese responded to t...,https://www.dailymail.com/news/article-1576611...,https://i.dailymail.com/1s/2026/04/26/05/10816...,2026-04-26T05:35:59Z,Prime Minister Anthony Albanese has said he is...
1,"{'id': 'abc-news-au', 'name': 'ABC News (AU)'}",Mya Kordic,Police charge 16-year-old with stabbing teen o...,A 16-year-old boy has been charged for alleged...,https://www.abc.net.au/news/2026-04-26/teenage...,https://live-production.wcms.abc-cdn.net.au/11...,2026-04-26T05:34:51Z,Police have charged a 16-year-old boy for alle...
2,"{'id': 'the-times-of-india', 'name': 'The Time...",AFP,"Chernobyl, 40 years since disaster: five thing...",Ukraine on Sunday marks the 40th anniversary o...,https://economictimes.indiatimes.com/news/inte...,"https://img.etimg.com/thumb/msid-130527091,wid...",2026-04-26T05:34:00Z,Ukraine on Sunday marks the 40th anniversary o...
3,"{'id': 'the-irish-times', 'name': 'The Irish T...",Keith Duggan,Donald Trump rushed from White House Correspon...,At least half a dozen shots were fired inside ...,https://www.irishtimes.com/world/us/2026/04/26...,https://www.irishtimes.com/resizer/v2/EZTDCX72...,2026-04-26T05:33:20Z,"Pandemonium, and a familiar queasiness, descen..."
4,"{'id': None, 'name': 'Freerepublic.com'}",My Catholic Life! (YouTube),[Catholic Caucus Devotional] My Catholic Life!...,Daily Readings from the USCCB“When he has driv...,https://freerepublic.com/focus/f-religion/4376...,NaN,2026-04-26T05:32:34Z,Skip to comments.\r\n[Catholic Caucus Devotion...


## 6. Clean and enrich articles

We select the useful fields, flatten the source field, build a combined text column, remove duplicates, and add country, category, and sentiment fields.

In [6]:
df = df[[
    "source",
    "author",
    "title",
    "description",
    "url",
    "publishedAt",
    "content",
]].copy()

df["source"] = df["source"].apply(lambda x: x["name"] if isinstance(x, dict) else x)
df["text"] = df["title"].fillna("") + " " + df["description"].fillna("")
df = df.drop_duplicates(subset=["title", "url"]).copy()

df["detected_countries"] = df["text"].apply(detect_countries)
df["category"] = df["text"].apply(classify_rule_based)
df["sentiment"] = df["text"].apply(sentiment)

df.to_csv("input_data.csv", index=False)

print(f"Saved {len(df)} classified articles to input_data.csv")
df.head()

Saved 93 classified articles to input_data.csv


,source,author,title,description,url,publishedAt,content,text,detected_countries,category,sentiment
0,Dailymail.com,Nicholas Comino,Albanese responds after man allegedly opens fi...,Prime Minister Anthony Albanese responded to t...,https://www.dailymail.com/news/article-1576611...,2026-04-26T05:35:59Z,Prime Minister Anthony Albanese has said he is...,Albanese responds after man allegedly opens fi...,[],Other,Neutral
1,ABC News (AU),Mya Kordic,Police charge 16-year-old with stabbing teen o...,A 16-year-old boy has been charged for alleged...,https://www.abc.net.au/news/2026-04-26/teenage...,2026-04-26T05:34:51Z,Police have charged a 16-year-old boy for alle...,Police charge 16-year-old with stabbing teen o...,[],Other,Negative
2,The Times of India,AFP,"Chernobyl, 40 years since disaster: five thing...",Ukraine on Sunday marks the 40th anniversary o...,https://economictimes.indiatimes.com/news/inte...,2026-04-26T05:34:00Z,Ukraine on Sunday marks the 40th anniversary o...,"Chernobyl, 40 years since disaster: five thing...",[Ukraine],Disaster,Negative
3,The Irish Times,Keith Duggan,Donald Trump rushed from White House Correspon...,At least half a dozen shots were fired inside ...,https://www.irishtimes.com/world/us/2026/04/26...,2026-04-26T05:33:20Z,"Pandemonium, and a familiar queasiness, descen...",Donald Trump rushed from White House Correspon...,[],Other,Neutral
4,Freerepublic.com,My Catholic Life! (YouTube),[Catholic Caucus Devotional] My Catholic Life!...,Daily Readings from the USCCB“When he has driv...,https://freerepublic.com/focus/f-religion/4376...,2026-04-26T05:32:34Z,Skip to comments.\r\n[Catholic Caucus Devotion...,[Catholic Caucus Devotional] My Catholic Life!...,[],Other,Neutral


## 7. Filter to relevant MSF countries

Only articles with at least one detected country from the MSF country list are kept. This creates the main dataset used by all models.

In [7]:
df = pd.read_csv("input_data.csv")
df["text"] = df["title"].fillna("") + " " + df["description"].fillna("")
df["detected_countries"] = df["text"].apply(detect_countries)

df = df[df["detected_countries"].apply(len) > 0].copy()

print("Category distribution:")
print(df["category"].value_counts(), "\n")

print("Sentiment distribution:")
print(df["sentiment"].value_counts(), "\n")

output_cols = [
    "source", "author", "title", "description", "url", "publishedAt",
    "detected_countries", "category", "sentiment"
]

df[output_cols].to_csv("msf_articles.csv", index=False)
df[output_cols].to_json("msf_articles.json", orient="records", indent=2)

df[output_cols].head()

Category distribution:
category
Disaster    2
Other       2
Conflict    2
Name: count, dtype: int64 

Sentiment distribution:
sentiment
Negative    3
Positive    2
Neutral     1
Name: count, dtype: int64 



,source,author,title,description,url,publishedAt,detected_countries,category,sentiment
2,The Times of India,AFP,"Chernobyl, 40 years since disaster: five thing...",Ukraine on Sunday marks the 40th anniversary o...,https://economictimes.indiatimes.com/news/inte...,2026-04-26T05:34:00Z,[Ukraine],Disaster,Negative
17,Antaranews.com,"Prisca Triferna, Resinta Sulistiyandari",Indonesia presses UN to probe peacekeeper kill...,The Indonesian government has reiterated its c...,https://en.antaranews.com/news/413637/indonesi...,2026-04-26T05:23:08Z,[Lebanon],Other,Positive
27,Freerepublic.com,Japan Times,Ukraine marks 40th anniversary of Chernobyl di...,Ukraine is commemorating the 40th anniversary ...,https://freerepublic.com/focus/f-chat/4376469/...,2026-04-26T05:11:42Z,[Ukraine],Conflict,Negative
31,Crypto Briefing,Estefano Gomez,Russia-Ukraine ceasefire unlikely as military ...,Continued military buildup suggests prolonged ...,https://cryptobriefing.com/russia-ukraine-ceas...,2026-04-26T05:04:20Z,[Ukraine],Conflict,Negative
42,The Irish Times,Jason Corcoran,I was playing Jenga with my young son in Dubli...,Though nothing compared with Ukraine’s sufferi...,https://www.irishtimes.com/life-style/people/2...,2026-04-26T05:00:00Z,[Ukraine],Other,Positive


## 8. Model 1: TF-IDF + Naive Bayes

This model converts article text into TF-IDF features and trains a Naive Bayes classifier. This is a strong, lightweight baseline for text classification.

In [8]:
df = pd.read_csv("msf_articles.csv")

if df.empty:
    raise ValueError("msf_articles.csv is empty.")

df["text"] = df["title"].fillna("") + " " + df["description"].fillna("")
df = df.drop_duplicates(subset=["title", "url"]).copy()

model_df = df[df["category"] != "Other"].copy()

print(f"Articles after deduplication: {len(df)}")
print("Training examples:", len(model_df))
print(model_df["category"].value_counts())

pipeline_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ("nb", MultinomialNB())
])

if len(model_df) == 0:
    print("No labelled examples for ML training. Falling back to existing category.")
    df["category_ml"] = df["category"]
else:
    X = model_df["text"]
    y = model_df["category"]
    min_class_count = y.value_counts().min()

    if len(model_df) >= 6 and min_class_count >= 2:
        n_splits = min(3, min_class_count)
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        cv_scores = cross_val_score(pipeline_tfidf, X, y, cv=cv, scoring="f1_macro")
        print("CV F1 scores:", cv_scores)
        print("Mean CV F1:", cv_scores.mean())

    if len(model_df) >= 10:
        X_train, X_test, y_train, y_test = safe_train_test_split(X, y)
        pipeline_tfidf.fit(X_train, y_train)
        y_pred = pipeline_tfidf.predict(X_test)
        print_model_report(y_test, y_pred, "TF-IDF + Naive Bayes")
    else:
        print("Not enough labelled data for train/test evaluation. Training on all labelled data.")
        pipeline_tfidf.fit(X, y)

    df["category_ml"] = pipeline_tfidf.predict(df["text"])

output_cols = [
    "source", "author", "title", "description", "url", "publishedAt",
    "detected_countries", "category", "category_ml", "sentiment"
]

df[output_cols].to_csv("msf_articles_tf_idf.csv", index=False)
df[output_cols].to_json("msf_articles_tf_idf.json", orient="records", indent=2)

df[output_cols].head()

Articles after deduplication: 6
Training examples: 4
category
Disaster    2
Conflict    2
Name: count, dtype: int64
Not enough labelled data for train/test evaluation. Training on all labelled data.


,source,author,title,description,url,publishedAt,detected_countries,category,category_ml,sentiment
0,The Times of India,AFP,"Chernobyl, 40 years since disaster: five thing...",Ukraine on Sunday marks the 40th anniversary o...,https://economictimes.indiatimes.com/news/inte...,2026-04-26T05:34:00Z,['Ukraine'],Disaster,Disaster,Negative
1,Antaranews.com,"Prisca Triferna, Resinta Sulistiyandari",Indonesia presses UN to probe peacekeeper kill...,The Indonesian government has reiterated its c...,https://en.antaranews.com/news/413637/indonesi...,2026-04-26T05:23:08Z,['Lebanon'],Other,Conflict,Positive
2,Freerepublic.com,Japan Times,Ukraine marks 40th anniversary of Chernobyl di...,Ukraine is commemorating the 40th anniversary ...,https://freerepublic.com/focus/f-chat/4376469/...,2026-04-26T05:11:42Z,['Ukraine'],Conflict,Conflict,Negative
3,Crypto Briefing,Estefano Gomez,Russia-Ukraine ceasefire unlikely as military ...,Continued military buildup suggests prolonged ...,https://cryptobriefing.com/russia-ukraine-ceas...,2026-04-26T05:04:20Z,['Ukraine'],Conflict,Conflict,Negative
4,The Irish Times,Jason Corcoran,I was playing Jenga with my young son in Dubli...,Though nothing compared with Ukraine’s sufferi...,https://www.irishtimes.com/life-style/people/2...,2026-04-26T05:00:00Z,['Ukraine'],Other,Conflict,Positive


## 9. Model 2: spaCy vectors + Logistic Regression

This model represents each article using spaCy document vectors, then trains Logistic Regression. With `en_core_web_sm`, vectors are limited, so this is mainly useful as a comparison. For better vector quality, use a larger spaCy model such as `en_core_web_md`.

In [9]:
df = pd.read_csv("msf_articles.csv")

if df.empty:
    raise ValueError("msf_articles.csv is empty.")

df["text"] = df["title"].fillna("") + " " + df["description"].fillna("")
model_df = df[df["category"] != "Other"].copy()

if model_df.empty:
    raise ValueError("No labelled examples available for training.")

X = np.vstack([nlp(text).vector for text in model_df["text"]])
y = model_df["category"]

class_counts = y.value_counts()

print("Class distribution:")
print(class_counts, "\n")

clf_spacy = LogisticRegression(max_iter=1000)

if len(model_df) < 10:
    print("Very small dataset — skipping train/test split.")
    print("Training on all available labelled data.\n")
    clf_spacy.fit(X, y)
else:
    min_class_count = class_counts.min()
    stratify = y if min_class_count >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify
    )

    clf_spacy.fit(X_train, y_train)
    y_pred = clf_spacy.predict(X_test)
    print_model_report(y_test, y_pred, "spaCy vectors + Logistic Regression")

X_all = np.vstack([nlp(text).vector for text in df["text"]])
df["category_spacy_vectors"] = clf_spacy.predict(X_all)

output_cols = [
    "source", "author", "title", "description", "url", "publishedAt",
    "detected_countries", "category", "category_spacy_vectors", "sentiment"
]

df[output_cols].to_csv("msf_articles_spacy_vectors.csv", index=False)
df[output_cols].to_json("msf_articles_spacy_vectors.json", orient="records", indent=2)

df[output_cols].head()

Class distribution:
category
Disaster    2
Conflict    2
Name: count, dtype: int64 

Very small dataset — skipping train/test split.
Training on all available labelled data.



,source,author,title,description,url,publishedAt,detected_countries,category,category_spacy_vectors,sentiment
0,The Times of India,AFP,"Chernobyl, 40 years since disaster: five thing...",Ukraine on Sunday marks the 40th anniversary o...,https://economictimes.indiatimes.com/news/inte...,2026-04-26T05:34:00Z,['Ukraine'],Disaster,Disaster,Negative
1,Antaranews.com,"Prisca Triferna, Resinta Sulistiyandari",Indonesia presses UN to probe peacekeeper kill...,The Indonesian government has reiterated its c...,https://en.antaranews.com/news/413637/indonesi...,2026-04-26T05:23:08Z,['Lebanon'],Other,Conflict,Positive
2,Freerepublic.com,Japan Times,Ukraine marks 40th anniversary of Chernobyl di...,Ukraine is commemorating the 40th anniversary ...,https://freerepublic.com/focus/f-chat/4376469/...,2026-04-26T05:11:42Z,['Ukraine'],Conflict,Conflict,Negative
3,Crypto Briefing,Estefano Gomez,Russia-Ukraine ceasefire unlikely as military ...,Continued military buildup suggests prolonged ...,https://cryptobriefing.com/russia-ukraine-ceas...,2026-04-26T05:04:20Z,['Ukraine'],Conflict,Conflict,Negative
4,The Irish Times,Jason Corcoran,I was playing Jenga with my young son in Dubli...,Though nothing compared with Ukraine’s sufferi...,https://www.irishtimes.com/life-style/people/2...,2026-04-26T05:00:00Z,['Ukraine'],Other,Conflict,Positive


## 10. Model 3: DistilBERT transformer classifier

This model fine-tunes `distilbert-base-uncased` for article category prediction. It is more powerful than the traditional models but slower and more data-hungry. If the dataset is very small, evaluation is skipped and the model trains on all available labelled examples.

In [10]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

df = pd.read_csv("msf_articles.csv")

if df.empty:
    raise ValueError("msf_articles.csv is empty.")

df["text"] = df["title"].fillna("") + " " + df["description"].fillna("")
model_df = df[df["category"] != "Other"].copy()

if model_df.empty:
    print("No labelled examples for transformer training. Falling back to existing category.")
    df["category_transformer"] = df["category"]
else:
    labels = sorted(model_df["category"].unique())
    label2id = {label: idx for idx, label in enumerate(labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    model_df["label"] = model_df["category"].map(label2id)

    print("Training examples:", len(model_df))
    print("Class distribution:")
    print(model_df["category"].value_counts(), "\n")

    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=256
        )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id
    )

    class_counts = model_df["label"].value_counts()
    min_class_count = class_counts.min()

    if len(model_df) < 10:
        print("Very small dataset — skipping train/test split.")
        print("Training transformer on all labelled data.\n")

        train_dataset = Dataset.from_pandas(model_df[["text", "label"]].reset_index(drop=True))
        train_dataset = train_dataset.map(tokenize, batched=True)
        train_dataset = train_dataset.remove_columns(["text"])
        train_dataset.set_format("torch")

        eval_dataset = None
    else:
        stratify = model_df["label"] if min_class_count >= 2 else None

        train_df, test_df = train_test_split(
            model_df[["text", "label"]],
            test_size=0.2,
            random_state=42,
            stratify=stratify
        )

        train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
        eval_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

        train_dataset = train_dataset.map(tokenize, batched=True)
        eval_dataset = eval_dataset.map(tokenize, batched=True)

        train_dataset = train_dataset.remove_columns(["text"])
        eval_dataset = eval_dataset.remove_columns(["text"])

        train_dataset.set_format("torch")
        eval_dataset.set_format("torch")

    training_args = TrainingArguments(
        output_dir="./transformer_results",
        eval_strategy="epoch" if eval_dataset is not None else "no",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=10,
        report_to="none",
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    trainer.train()

    if eval_dataset is not None:
        predictions = trainer.predict(eval_dataset)

        y_test = predictions.label_ids
        y_pred = np.argmax(predictions.predictions, axis=1)
        target_names = [id2label[i] for i in range(len(labels))]

        print("Classification report:")
        print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))
        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))
    else:
        print("Evaluation skipped because dataset is too small.")

    full_dataset = Dataset.from_pandas(df[["text"]].reset_index(drop=True))
    full_dataset = full_dataset.map(tokenize, batched=True)
    full_dataset = full_dataset.remove_columns(["text"])
    full_dataset.set_format("torch")

    full_predictions = trainer.predict(full_dataset)
    full_pred_ids = np.argmax(full_predictions.predictions, axis=1)

    df["category_transformer"] = [id2label[i] for i in full_pred_ids]

output_cols = [
    "source", "author", "title", "description", "url", "publishedAt",
    "detected_countries", "category", "category_transformer", "sentiment"
]

df[output_cols].to_csv("msf_articles_transformer.csv", index=False)
df[output_cols].to_json("msf_articles_transformer.json", orient="records", indent=2)

df[output_cols].head()

d:\William\Documents\Python\msf_news_monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training examples: 4
Class distribution:
category
Disaster    2
Conflict    2
Name: count, dtype: int64 



Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2789.84it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Very small dataset — skipping train/test split.
Training transformer on all labelled data.



Map: 100%|██████████| 4/4 [00:00<00:00, 133.00 examples/s]


Step,Training Loss


Evaluation skipped because dataset is too small.


Map: 100%|██████████| 6/6 [00:00<00:00, 660.73 examples/s]


,source,author,title,description,url,publishedAt,detected_countries,category,category_transformer,sentiment
0,The Times of India,AFP,"Chernobyl, 40 years since disaster: five thing...",Ukraine on Sunday marks the 40th anniversary o...,https://economictimes.indiatimes.com/news/inte...,2026-04-26T05:34:00Z,['Ukraine'],Disaster,Disaster,Negative
1,Antaranews.com,"Prisca Triferna, Resinta Sulistiyandari",Indonesia presses UN to probe peacekeeper kill...,The Indonesian government has reiterated its c...,https://en.antaranews.com/news/413637/indonesi...,2026-04-26T05:23:08Z,['Lebanon'],Other,Conflict,Positive
2,Freerepublic.com,Japan Times,Ukraine marks 40th anniversary of Chernobyl di...,Ukraine is commemorating the 40th anniversary ...,https://freerepublic.com/focus/f-chat/4376469/...,2026-04-26T05:11:42Z,['Ukraine'],Conflict,Conflict,Negative
3,Crypto Briefing,Estefano Gomez,Russia-Ukraine ceasefire unlikely as military ...,Continued military buildup suggests prolonged ...,https://cryptobriefing.com/russia-ukraine-ceas...,2026-04-26T05:04:20Z,['Ukraine'],Conflict,Conflict,Negative
4,The Irish Times,Jason Corcoran,I was playing Jenga with my young son in Dubli...,Though nothing compared with Ukraine’s sufferi...,https://www.irishtimes.com/life-style/people/2...,2026-04-26T05:00:00Z,['Ukraine'],Other,Conflict,Positive


## 11. Final comparison table

This section combines the outputs from all models into one dataframe so predictions can be compared side by side.

In [11]:
base_df = pd.read_csv("msf_articles.csv")

tfidf_df = pd.read_csv("msf_articles_tf_idf.csv")
spacy_df = pd.read_csv("msf_articles_spacy_vectors.csv")
transformer_df = pd.read_csv("msf_articles_transformer.csv")

comparison_df = base_df.copy()
comparison_df["category_ml"] = tfidf_df["category_ml"]
comparison_df["category_spacy_vectors"] = spacy_df["category_spacy_vectors"]
comparison_df["category_transformer"] = transformer_df["category_transformer"]

comparison_cols = [
    "source", "title", "detected_countries", "category",
    "category_ml", "category_spacy_vectors", "category_transformer", "sentiment", "url"
]

comparison_df[comparison_cols].to_csv("msf_articles_model_comparison.csv", index=False)
comparison_df[comparison_cols].to_json("msf_articles_model_comparison.json", orient="records", indent=2)

comparison_df[comparison_cols].head()

,source,title,detected_countries,category,category_ml,category_spacy_vectors,category_transformer,sentiment,url
0,The Times of India,"Chernobyl, 40 years since disaster: five thing...",['Ukraine'],Disaster,Disaster,Disaster,Disaster,Negative,https://economictimes.indiatimes.com/news/inte...
1,Antaranews.com,Indonesia presses UN to probe peacekeeper kill...,['Lebanon'],Other,Conflict,Conflict,Conflict,Positive,https://en.antaranews.com/news/413637/indonesi...
2,Freerepublic.com,Ukraine marks 40th anniversary of Chernobyl di...,['Ukraine'],Conflict,Conflict,Conflict,Conflict,Negative,https://freerepublic.com/focus/f-chat/4376469/...
3,Crypto Briefing,Russia-Ukraine ceasefire unlikely as military ...,['Ukraine'],Conflict,Conflict,Conflict,Conflict,Negative,https://cryptobriefing.com/russia-ukraine-ceas...
4,The Irish Times,I was playing Jenga with my young son in Dubli...,['Ukraine'],Other,Conflict,Conflict,Conflict,Positive,https://www.irishtimes.com/life-style/people/2...
